# 1 · Exploración inicial del dataset

**Proyecto:** Predicción de readmisión hospitalaria en pacientes diabéticos
**Dataset:** *Diabetes 130-US hospitals for years 1999-2008* (UCI Machine Learning Repository)
**Autor:** Diego Rodríguez Díaz del Campo

---

### Objetivo de este notebook

Conocer el dataset **antes de tomar ninguna decisión sobre él**. Aquí no se limpia:
se observa, se mide y se documenta. Los únicos cambios que se aplican al fichero son dos
normalizaciones de valores desconocidos: la sustitución del carácter `?` por `NaN` (sin
ella Pandas no puede analizar correctamente los valores ausentes) y la conversión de la
marca `Unknown/Invalid` de `gender` a `NaN`, para que ambos casos reciban el mismo
tratamiento en la limpieza.

Todas las decisiones de limpieza que se detecten aquí se **justifican en las conclusiones**
y se **aplican en el notebook 02**, nunca en este.

| | |
|---|---|
| **Entrada** | `data/raw/diabetic_data.csv` (101.766 × 50) |
| **Salida** | `data/processed/diabetic_data_replaceNaN.csv` |
| **Filas eliminadas** | Ninguna |
| **Cambios aplicados** | `?` → `NaN` en todo el dataset; `Unknown/Invalid` → `NaN` en `gender` |

### Contenido

1. Configuración del entorno y rutas
2. Carga y estructura general
3. Distribución de la variable objetivo y de las variables demográficas
4. Tratamiento del valor `?`
5. Naturaleza de cada variable
6. Variables constantes y registros duplicados
7. Consistencia de las categorías
8. Rangos de las variables numéricas
9. Exportación
10. Conclusiones

## 1 · Configuración del entorno y rutas

Las rutas se definen **una sola vez** con `pathlib` en lugar de escribir la ruta
absoluta en cada celda. Así, si el proyecto cambia de ubicación, basta con
modificar una única línea.

> **Convención del proyecto:** `data/raw/` contiene el dataset original tal y como
> se descargó y **nunca se escribe nada en él**. Todo lo que se genera va a
> `data/processed/`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(r'D:\Cosas Diego\Universidad\Proyectos personales\proyecto 1')
RAW = BASE / 'data' / 'raw'
PROC = BASE / 'data' / 'processed'

## 2 · Carga y estructura general

Primer contacto con los datos: cuántos registros hay, qué variables los describen,
de qué tipo son y cuánta variedad contiene cada una.

Cada comprobación va en su propia celda: si se agrupan varias en una sola, Jupyter
solo muestra el resultado de la última y el resto del análisis se pierde.

In [2]:
# low_memory=False evita el DtypeWarning: sin él, Pandas lee el fichero por
# bloques y puede asignar tipos distintos a la misma columna en bloques distintos.

df = pd.read_csv(RAW / 'diabetic_data.csv', low_memory=False)

print(f'Filas: {df.shape[0]}   Columnas: {df.shape[1]}')

Filas: 101766   Columnas: 50


In [3]:
# Tipos de dato y memoria ocupada por cada variable
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [4]:
# Resumen estadístico de las variables almacenadas como numéricas.
# .T transpone la tabla para que cada variable sea una fila y se lea mejor.
df.describe().T

,count,mean,std,min,25%,50%,75%,max
encounter_id,101766.0,1.652016e+08,1.026403e+08,12522.0,84961194.0,152388987.0,2.302709e+08,443867222.0
patient_nbr,101766.0,5.433040e+07,3.869636e+07,135.0,23413221.0,45505143.0,8.754595e+07,189502619.0
admission_type_id,101766.0,2.024006e+00,1.445403e+00,1.0,1.0,1.0,3.000000e+00,8.0
discharge_disposition_id,101766.0,3.715642e+00,5.280166e+00,1.0,1.0,1.0,4.000000e+00,28.0
admission_source_id,101766.0,5.754437e+00,4.064081e+00,1.0,1.0,7.0,7.000000e+00,25.0
time_in_hospital,101766.0,4.395987e+00,2.985108e+00,1.0,2.0,4.0,6.000000e+00,14.0
num_lab_procedures,101766.0,4.309564e+01,1.967436e+01,1.0,31.0,44.0,5.700000e+01,132.0
num_procedures,101766.0,1.339730e+00,1.705807e+00,0.0,0.0,1.0,2.000000e+00,6.0
num_medications,101766.0,1.602184e+01,8.127566e+00,1.0,10.0,15.0,2.000000e+01,81.0
number_outpatient,101766.0,3.693572e-01,1.267265e+00,0.0,0.0,0.0,0.000000e+00,42.0


In [5]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [6]:
# Número de valores distintos por variable. Es la primera pista sobre qué
# variables son identificadores, cuáles son categóricas y cuáles constantes.
df.nunique()

encounter_id                101766
patient_nbr                  71518
race                             6
gender                           3
age                             10
weight                          10
admission_type_id                8
discharge_disposition_id        26
admission_source_id             17
time_in_hospital                14
payer_code                      18
medical_specialty               73
num_lab_procedures             118
num_procedures                   7
num_medications                 75
number_outpatient               39
number_emergency                33
number_inpatient                21
diag_1                         717
diag_2                         749
diag_3                         790
number_diagnoses                16
max_glu_serum                    3
A1Cresult                        3
metformin                        4
repaglinide                      4
nateglinide                      4
chlorpropamide                   4
glimepiride         

## 3 · Variable objetivo y variables demográficas

`readmitted` es la variable que el proyecto quiere predecir. Sus tres categorías son:

| Valor | Significado |
|---|---|
| `NO` | El paciente no fue readmitido |
| `>30` | Readmitido **después** de 30 días |
| `<30` | Readmitido **antes** de 30 días |

Se revisan también `gender` y `race` porque son las variables donde con más
frecuencia aparecen categorías mal codificadas.

In [7]:
print(df['readmitted'].value_counts(), end='\n\n')
print(df['gender'].value_counts(dropna=False), end='\n\n')
print(df['race'].value_counts(dropna=False))

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

race
Caucasian          76099
AfricanAmerican    19210
?                   2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64


## 4 · Tratamiento del valor `?`

El dataset original no usa celdas vacías para la información desconocida: usa el
carácter `?`. Para Pandas eso es texto normal, así que `isna()` devolvería cero y
todo el análisis de valores faltantes sería erróneo.

El procedimiento es: **primero medir** cuántos `?` hay en cada columna y **después**
sustituirlos por `NaN`, para que Pandas los reconozca como ausentes.

In [8]:
# Conteo de '?' por columna, antes de sustituir nada
missing_data = (df == '?').sum().sort_values(ascending=False)
missing_data_percentage = (missing_data / len(df)) * 100

print('Porcentaje de valores "?" por columna:')
print(missing_data_percentage[missing_data_percentage > 0])

Porcentaje de valores "?" por columna:
weight               96.858479
medical_specialty    49.082208
payer_code           39.557416
race                  2.233555
diag_3                1.398306
diag_2                0.351787
diag_1                0.020636
dtype: float64


In [9]:
# Sustitución de '?' por NaN en todo el DataFrame
df = df.replace('?', np.nan)

nan_percentage = df.isna().sum() / len(df) * 100

print('Porcentaje de NaN tras la sustitución:')
print(nan_percentage[nan_percentage > 0].sort_values(ascending=False))

Porcentaje de NaN tras la sustitución:
weight               96.858479
max_glu_serum        94.746772
A1Cresult            83.277322
medical_specialty    49.082208
payer_code           39.557416
race                  2.233555
diag_3                1.398306
diag_2                0.351787
diag_1                0.020636
dtype: float64


### 4.1 · Un matiz importante: `max_glu_serum` y `A1Cresult`

Estas dos columnas aparecen con un porcentaje altísimo de `NaN`, pero **no son datos
perdidos**.

En el CSV original su valor es la cadena literal `None` (96.420 y 84.748 filas
respectivamente). Pandas la convierte automáticamente a `NaN` al leer el fichero,
porque `"None"` forma parte de su lista por defecto de valores nulos.

Es decir: el dataset original **sí sabía** el valor, y ese valor significa
*"la prueba no se realizó"*.

La diferencia es clínica, no técnica. Que un médico decida no pedir una hemoglobina
glicosilada es en sí mismo un dato sobre el manejo del paciente. Estas variables no
deben imputarse, sino **recuperar su significado original** con una categoría explícita.

## 5 · Naturaleza de cada variable

Antes de decidir qué se elimina, hay que entender **qué representa** cada columna.
Se revisan todos los valores de todas las variables y, a partir de esa lectura, se
construye un diccionario que clasifica cada una según su naturaleza real.

Este diccionario es la referencia del proyecto: distingue lo que Pandas *cree* que es
una variable (`int64`, `object`) de lo que la variable *es* (un identificador, un
código categórico, una magnitud).

In [10]:
# Revisión de los valores de cada columna
for column in df.columns:
    print(df[column].value_counts(ascending=False, dropna=False), end='\n\n')

encounter_id
2278392      1
190792044    1
190790070    1
190789722    1
190786806    1
            ..
106665324    1
106657776    1
106644876    1
106644474    1
443867222    1
Name: count, Length: 101766, dtype: int64

patient_nbr
88785891     40
43140906     28
1660293      23
88227540     23
23199021     23
             ..
11005362      1
98252496      1
1019673       1
13396320      1
175429310     1
Name: count, Length: 71518, dtype: int64

race
Caucasian          76099
AfricanAmerican    19210
NaN                 2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64

gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

age
[70-80)     26068
[60-70)     22483
[50-60)     17256
[80-90)     17197
[40-50)      9685
[30-40)      3775
[90-100)     2793
[20-30)      1657
[10-20)       691
[0-10)        161
Name: count, dtype: int64

weight
NaN          98569
[75-100)      1336
[5

In [11]:
variable_types = {
    # Identificadores
    'encounter_id': 'identifier',
    'patient_nbr': 'identifier',

    # Variables demográficas
    'race': 'categorical',
    'gender': 'categorical',
    'age': 'ordinal_categorical',
    'weight': 'ordinal_categorical',

    # Información administrativa
    'admission_type_id': 'categorical_code',
    'discharge_disposition_id': 'categorical_code',
    'admission_source_id': 'categorical_code',
    'payer_code': 'categorical',
    'medical_specialty': 'categorical',

    # Información clínica / hospitalaria
    'time_in_hospital': 'numeric',
    'num_lab_procedures': 'numeric',
    'num_procedures': 'numeric',
    'num_medications': 'numeric',
    'number_outpatient': 'numeric',
    'number_emergency': 'numeric',
    'number_inpatient': 'numeric',
    'number_diagnoses': 'numeric',

    # Diagnósticos
    'diag_1': 'diagnosis_code',
    'diag_2': 'diagnosis_code',
    'diag_3': 'diagnosis_code',

    # Resultados de pruebas (ordinales: Norm < >200 < >300 y Norm < >7 < >8)
    'max_glu_serum': 'ordinal_categorical',
    'A1Cresult': 'ordinal_categorical',

    # Medicación
    'metformin': 'categorical',
    'repaglinide': 'categorical',
    'nateglinide': 'categorical',
    'chlorpropamide': 'categorical',
    'glimepiride': 'categorical',
    'acetohexamide': 'categorical',
    'glipizide': 'categorical',
    'glyburide': 'categorical',
    'tolbutamide': 'categorical',
    'pioglitazone': 'categorical',
    'rosiglitazone': 'categorical',
    'acarbose': 'categorical',
    'miglitol': 'categorical',
    'troglitazone': 'categorical',
    'tolazamide': 'categorical',
    'examide': 'categorical',
    'citoglipton': 'categorical',
    'insulin': 'categorical',
    'glyburide-metformin': 'categorical',
    'glipizide-metformin': 'categorical',
    'glimepiride-pioglitazone': 'categorical',
    'metformin-rosiglitazone': 'categorical',
    'metformin-pioglitazone': 'categorical',

    # Cambios / tratamiento de diabetes
    'change': 'categorical',
    'diabetesMed': 'categorical',

    # Variable objetivo
    'readmitted': 'target'
}

print(f'Variables clasificadas: {len(variable_types)}')

Variables clasificadas: 50


## 6 · Variables constantes y registros duplicados

Una variable en la que **todos los pacientes tienen el mismo valor** no puede explicar
por qué unos son readmitidos y otros no: no distingue a nadie de nadie. Ordenar las
variables por número de valores únicos las deja al descubierto de inmediato.

In [12]:
df.nunique(dropna=False).sort_values(ascending=True)

examide                          1
citoglipton                      1
troglitazone                     2
acetohexamide                    2
tolbutamide                      2
glipizide-metformin              2
diabetesMed                      2
metformin-rosiglitazone          2
change                           2
glimepiride-pioglitazone         2
metformin-pioglitazone           2
readmitted                       3
gender                           3
tolazamide                       3
chlorpropamide                   4
glimepiride                      4
glipizide                        4
glyburide                        4
glyburide-metformin              4
acarbose                         4
miglitol                         4
nateglinide                      4
insulin                          4
pioglitazone                     4
rosiglitazone                    4
metformin                        4
A1Cresult                        4
max_glu_serum                    4
repaglinide         

Dos variables aparecen con **un único valor**:

| Variable | Valores únicos | Contenido |
|---|---|---|
| `examide` | 1 | Todos `"No"` |
| `citoglipton` | 1 | Todos `"No"` |

Son constantes: no aportan ninguna información y son candidatas a eliminación.

In [13]:
# Filas exactamente idénticas en todas sus columnas
print(f'Número de filas duplicadas: {df.duplicated().sum()}')

Número de filas duplicadas: 0


## 7 · Consistencia de las categorías

Los valores de todas las variables de texto ya se han revisado a ojo en la sección 5. Aquí
se comprueba **de forma programática** el problema más habitual en variables de texto: dos
categorías que en realidad son la misma pero difieren por un espacio sobrante o por
mayúsculas (`'Female'` y `'female '`, por ejemplo). Pandas las contaría como distintas y el
recuento de categorías quedaría inflado.

La comprobación consiste en contar las categorías de cada variable antes y después de
normalizar el texto (quitar espacios y pasar a minúsculas). Si el número no cambia, no hay
categorías duplicadas por escritura.

In [14]:
# .str.strip() quita espacios al principio y al final; .str.lower() pasa a minúsculas.
# Si al normalizar se funden categorías, nunique() baja y la variable se imprime.
for column in df.select_dtypes(include='object').columns:
    original    = df[column].nunique()
    normalizado = df[column].str.strip().str.lower().nunique()

    if original != normalizado:
        print(f'{column}: {original} categorías -> {normalizado} tras normalizar')

print('Comprobación terminada.')

Comprobación terminada.


No se detecta ninguna categoría duplicada por escritura: en las 37 variables de texto, el
número de categorías es el mismo antes y después de normalizar.

La única categoría problemática es de otro tipo y ya apareció en la sección 3: `gender`
contiene `Unknown/Invalid` en 3 registros, que no es un género sino una marca de dato no
válido. Se convierte a `NaN` para que reciba el mismo tratamiento que el resto de valores
desconocidos durante la limpieza.

In [15]:
df['gender'] = df['gender'].replace('Unknown/Invalid', np.nan)

print(df['gender'].value_counts(dropna=False))

gender
Female    54708
Male      47055
NaN           3
Name: count, dtype: int64


## 8 · Rangos de las variables numéricas

Última comprobación antes de exportar: verificar que las magnitudes numéricas no
contienen valores imposibles (negativos en recuentos, duraciones desproporcionadas,
etc.) que delatarían errores de registro.

Las variables se seleccionan **desde el diccionario de la sección 5**, no por su `dtype`:
`select_dtypes` devolvería también los identificadores y los códigos `*_id`, que son
enteros pero no magnitudes, y su mínimo y máximo no significan nada.

In [16]:
# Del diccionario, solo las columnas clasificadas como 'numeric'
columnas_numericas = [col for col, tipo in variable_types.items() if tipo == 'numeric']

# .agg(['min', 'max']) aplica las dos funciones a cada columna; .T deja una variable por fila
df[columnas_numericas].agg(['min', 'max']).T

,min,max
time_in_hospital,1,14
num_lab_procedures,1,132
num_procedures,0,6
num_medications,1,81
number_outpatient,0,42
number_emergency,0,76
number_inpatient,0,21
number_diagnoses,1,16


Las ocho magnitudes se mueven en rangos clínicamente plausibles: estancias de 1 a 14 días,
hasta 132 pruebas de laboratorio, hasta 81 medicaciones y de 1 a 16 diagnósticos. Los
recuentos de visitas previas llegan a 42, 76 y 21, valores altos pero posibles en pacientes
crónicos con uso intensivo del sistema sanitario. **No se corrige ninguna.**

Queda también ilustrado el motivo de haber construido el diccionario: hay columnas
almacenadas como `int64` que no son magnitudes —identificadores y códigos `*_id`— y el tipo
que asigna Pandas no determina la naturaleza de la variable.

## 9 · Exportación

Se guarda el dataset con los `?` y el `Unknown/Invalid` de `gender` ya convertidos a
`NaN`, que será el punto de partida del notebook de limpieza. El fichero original permanece intacto en `data/raw/`.

> **Nota:** `to_csv()` no devuelve un DataFrame, devuelve `None`. Por eso se escribe
> `df.to_csv(...)` y **nunca** `df = df.to_csv(...)`: esto último dejaría `df` valiendo
> `None` y rompería cualquier celda que se ejecutase después.

In [17]:
df.to_csv(PROC / 'diabetic_data_replaceNaN.csv', index=False)

print(f'Guardado en: {PROC / "diabetic_data_replaceNaN.csv"}')
print(f'Dimensiones: {df.shape}')

Guardado en: D:\Cosas Diego\Universidad\Proyectos personales\proyecto 1\data\processed\diabetic_data_replaceNaN.csv
Dimensiones: (101766, 50)


---

## 10 · Conclusiones de la exploración

### Resumen del dataset

| Concepto | Valor |
|---|---|
| Registros (ingresos hospitalarios) | 101.766 |
| Variables | 50 |
| Pacientes distintos | 71.518 |
| Filas duplicadas | 0 |
| Variable objetivo | `readmitted` (`NO` / `>30` / `<30`) |

### Decisiones que se aplicarán en el notebook 02

| Variable | Situación observada | Decisión | Motivo |
|---|---|---|---|
| `weight` | ~97% desconocido | **Eliminar** | Información insuficiente para ser utilizable |
| `examide`, `citoglipton` | Un único valor | **Eliminar** | Constantes: no aportan variabilidad |
| `encounter_id`, `patient_nbr` | Identificadores | **No usar como predictores** | No son características clínicas |
| `medical_specialty` | ~49% desconocido | Categoría `Unknown` | Imputar la moda en la mitad del dataset sería inventar información |
| `payer_code` | ~40% desconocido | Categoría `Unknown` | Mismo criterio |
| `race` | ~2,2% desconocido | Categoría `Unknown` | La ausencia depende del hospital y del registro administrativo: no es aleatoria, y por tanto es informativa |
| `gender` | 3 valores `Unknown/Invalid` | Imputar con la moda | Cantidad insignificante frente a 101.766 registros |
| `diag_1`, `diag_2`, `diag_3` | <1,5% desconocido | Conservar `NaN` | Asignar un diagnóstico artificial no tiene sentido clínico |
| `max_glu_serum`, `A1Cresult` | Cadena `None` leída como `NaN` | Categoría `Not_Measured` | No falta el dato: el dato **es** que la prueba no se hizo |
| Códigos `*_id` | Almacenados como `int64` | Convertir a `category` | Son códigos, no magnitudes |

### Observaciones que condicionan el resto del proyecto

- **El dataset contiene 101.766 ingresos, pero solo 71.518 pacientes distintos.**
  Un mismo paciente aparece en varias filas. Esto no afecta a la limpieza, pero será
  determinante al separar entrenamiento y test: si distintos ingresos del mismo
  paciente caen a ambos lados de la partición, el modelo lo reconocerá y los
  resultados quedarán artificialmente inflados. `patient_nbr` deberá recuperarse en
  ese momento.

- **No hay valores imposibles en las variables numéricas.** Los valores extremos
  observados (por ejemplo, pacientes con muchos ingresos previos) son clínicamente
  plausibles y no se tratarán como errores.

### Siguiente paso

Aplicar estas decisiones de forma controlada y documentada en
**`02_limpieza_dataset.ipynb`**.